# 12.6 Real PyTorch Geometric: data, batches, and message-passing layers

**Research task:** a collaborator gives you molecular structures and one property per molecule. Before training, you must prove that the graph library preserves the structures, labels, and message directions. This lesson implements that check with **PyTorch Geometric (PyG)**, not a substitute interface.

## Start here

You need the ideas of an atom feature, a directed edge, and a graph readout from [12.1](Chapter12_Part1.ipynb) and [12.2](Chapter12_Part2.ipynb). A tensor is simply an array with a shape and data type; `x[2]` is the feature row of atom 2. A **mini-batch** is a collection of complete molecules processed together.

By the end you will create `Data`, `Batch` and `DataLoader` objects, implement `MessagePassing`, use real `GCNConv` and `GINEConv` layers, and verify their operations against small calculations. The networks here are untrained; the next lesson fits a measured property.

**Setup:** update the [course environment](Readme.md#set-up-python), which pins tested PyG 2.8.0.post1. These examples need only the standard PyG package and existing CPU PyTorch. They do not require `torch-scatter`, `torch-sparse`, `pyg-lib`, a GPU, or any dataset download. Installation is a terminal step; code cells do not install packages. See the [official installation guide](https://pytorch-geometric.readthedocs.io/en/2.8.0/install/installation.html).

**Learning route:** read the short equations and predict the printed shapes before running each cell. The custom-layer proof is a deeper exercise; the `Data`/batching and built-in-layer sections are the core practical skills.

In [ ]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
import torch
from torch import nn
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, GCNConv, GINEConv, global_mean_pool
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Scaffolds import MurckoScaffold

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
SEED = 2026
torch.manual_seed(SEED)
OUT = Path('outputs/chapter12_part6')
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {'torch': str(torch.__version__), 'torch_geometric': torch_geometric.__version__,
            'rdkit': rdBase.rdkitVersion, 'numpy': np.__version__}
print(VERSIONS)
print('CPU only; no compiled graph extensions or dataset downloads.')

## 12.6.1 One molecule becomes one `Data` object

| Field | Shape for one molecule | Meaning |
| --- | --- | --- |
| `x` | `(atoms, 17)` | Element category, degree, H count, charge and aromatic/ring flags |
| `edge_index` | `(2, directed_edges)` | Source row first, destination row second |
| `edge_attr` | `(directed_edges, 7)` | Bond type, conjugation and ring membership |
| `y` | `(1,)` | One graph-level target, when supplied |
| `source_row` | `(1,)` | Original record identity; bookkeeping, never a predictor |

We use the compact encoding from 12.3: element identity is one-hot with an `other` category; degree and H count are divided by 4, charge by 2. These divisors are fixed definitions, not fitted scaling. Formal charges are retained. Stereo, isotopes, radical state, coordinates and experimental context are omitted. Part 12.1 demonstrates richer stereo-aware features; columns from different encoders must not be mixed.

The default RDKit parser normally removes ordinary explicit H nodes; attached H counts are included in the atom features. Connected records only are accepted. This is a declared teaching policy, not universal molecular standardization. PyG validates tensor addresses, not chemical correctness. [PyG `Data` tutorial](https://pytorch-geometric.readthedocs.io/en/2.8.0/get_started/introduction.html).

In [ ]:
ELEMENTS = [5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
NODE_NAMES = [f'element_{z}' for z in ELEMENTS] + ['element_other', 'degree/4',
              'attached_H/4', 'formal_charge/2', 'aromatic', 'in_ring']
EDGE_NAMES = ['single', 'double', 'triple', 'aromatic', 'other', 'conjugated', 'in_ring']
FEATURE_SCHEMA = {'elements': ELEMENTS, 'node_names': NODE_NAMES, 'edge_names': EDGE_NAMES,
    'hydrogens': 'RDKit default implicit H; attached counts on atoms',
    'components': 'connected only', 'stereochemistry': 'omitted', 'coordinates': 'omitted'}

def one_hot_other(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def mol_to_data(mol, target=None, source_row=None):
    if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError('Provide a nonempty connected molecule.')
    nodes = [one_hot_other(a.GetAtomicNum(), ELEMENTS) +
             [a.GetDegree()/4, a.GetTotalNumHs()/4, a.GetFormalCharge()/2,
              float(a.GetIsAromatic()), float(a.IsInRing())] for a in mol.GetAtoms()]
    edges, attributes = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        attr = one_hot_other(b.GetBondType(), BOND_TYPES) + [float(b.GetIsConjugated()), float(b.IsInRing())]
        edges.extend([(i, j), (j, i)])
        attributes.extend([attr, attr])
    data = Data(x=torch.tensor(nodes, dtype=torch.float32),
        edge_index=torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T.contiguous(),
        edge_attr=torch.tensor(attributes, dtype=torch.float32).reshape(-1, len(EDGE_NAMES)),
        num_nodes=mol.GetNumAtoms())
    if target is not None:
        data.y = torch.tensor([target], dtype=torch.float32)
    if source_row is not None:
        data.source_row = torch.tensor([source_row], dtype=torch.long)
    data.validate(raise_on_error=True)
    return data

In [ ]:
names = ['ethanol', 'water', 'acetic acid']
smiles = ['CCO', 'O', 'CC(=O)O']
molecules = [Chem.MolFromSmiles(s) for s in smiles]
graphs = [mol_to_data(m, target=Descriptors.MolWt(m), source_row=100+i) for i, m in enumerate(molecules)]
print(graphs[0])
display(pd.DataFrame([{'name': n, 'nodes': g.num_nodes, 'directed_edges': g.num_edges,
                      'calculated_MolWt': float(g.y[0]), 'source_row': int(g.source_row[0])}
                     for n, g in zip(names, graphs)]))
assert graphs[1].edge_index.shape == (2, 0) and graphs[1].edge_attr.shape == (0, 7)
drawer = rdMolDraw2D.MolDraw2DCairo(960, 250, 320, 250)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecules(molecules, legends=names)
drawer.FinishDrawing()
png = drawer.GetDrawingText()
(OUT / 'data_molecules.png').write_bytes(png)
display(Image(data=png))
print('The masses here are calculated descriptors used to check label routing, not experimental training data.')

## 12.6.2 The batch is a disjoint union

Ethanol has 3 nodes, water 1, and acetic acid 4. The combined `x` has 8 rows. Acid's local atom 0 moves to global row 4; PyG adds this offset to its edge addresses. `batch` identifies which molecule owns each row; `ptr` gives the start and end boundaries. Targets remain one per molecule.

**Predict:** the boundaries should be `[0, 3, 4, 8]`, and water contributes no edge columns. Does a sodium ion need an edge to remain in a batch? No: an isolated node is still a graph.

PyG uses attribute names to decide some batching increments. Names containing `index` receive special treatment by default; use an explicit metadata name such as `source_row`, or define/test custom batching rules. Always verify round-trip record identities. [Batching documentation](https://pytorch-geometric.readthedocs.io/en/2.8.0/advanced/batching.html).

In [ ]:
batch = Batch.from_data_list(graphs)
print(batch)
print('ptr:', batch.ptr.tolist(), '| node membership:', batch.batch.tolist())
assert batch.ptr.tolist() == [0, 3, 4, 8]
assert batch.source_row.tolist() == [100, 101, 102]
assert batch.y.shape == (3,)
src, dst = batch.edge_index
assert torch.equal(batch.batch[src], batch.batch[dst])
recovered = batch.to_data_list()
for original, restored in zip(graphs, recovered):
    for key in ['x', 'edge_index', 'edge_attr', 'y', 'source_row']:
        torch.testing.assert_close(original[key], restored[key])

adjacency = np.zeros((batch.num_nodes, batch.num_nodes))
adjacency[dst.numpy(), src.numpy()] = 1
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout='constrained')
axes[0].imshow(adjacency, cmap='Blues', vmin=0, vmax=1)
axes[0].set(xlabel='Source node', ylabel='Destination node', title='Actual PyG batch connectivity')
for boundary in batch.ptr[1:-1]:
    axes[0].axhline(int(boundary)-0.5, color='orange', lw=1)
    axes[0].axvline(int(boundary)-0.5, color='orange', lw=1)
axes[1].bar(np.arange(8), batch.batch.numpy()+1, color=['#247e9f']*3+['#d88b2b']+['#8064a2']*4)
axes[1].set(xlabel='Global atom row', ylabel='Graph number + 1', title='Membership routes the readout', yticks=[1,2,3])
fig.savefig(OUT / 'pyg_batch.png', dpi=140)
plt.show()

## 12.6.3 `DataLoader` batches whole molecules

`batch_size=2` means two graphs, not two atoms. Different batches may have different numbers of nodes and edges. `shuffle=True` changes graph order during training; it must move each graph's target with it. For a small CPU lesson, `num_workers=0` avoids background worker complexity. A fixed generator seeds the shuffle.

This minibatch split has nothing to do with the train/validation/test partition. Decide which molecules belong in each partition first, then make a loader for each partition.

In [ ]:
loader = DataLoader(graphs, batch_size=2, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(SEED))
seen = []
for mini_batch in loader:
    print('graph IDs:', mini_batch.source_row.tolist(), 'x shape:', tuple(mini_batch.x.shape),
          'y shape:', tuple(mini_batch.y.shape))
    assert mini_batch.num_graphs == len(mini_batch.y)
    for row, value in zip(mini_batch.source_row.tolist(), mini_batch.y.tolist()):
        assert np.isclose(value, float(graphs[row-100].y[0]))
        seen.append(row)
assert sorted(seen) == [100, 101, 102]
pooled = global_mean_pool(batch.x, batch.batch)
assert pooled.shape == (3, len(NODE_NAMES))
print('Mean readout shape:', tuple(pooled.shape))

## 12.6.4 Implement a real `MessagePassing` layer

PyG supplies the routing and reduction. We supply the message function:

$$
h_i'=W_s x_i+b_s+\sum_{j\in\mathcal N(i)}\operatorname{ReLU}(W_nx_j+W_e e_{ji}).
$$

Here $x_i$ is atom $i$'s input row, $e_{ji}$ describes a bond into atom $i$, and each $W$ is a learned linear map. The same maps are reused for every atom/bond. With `flow='source_to_target'`, an argument named `x_j` automatically selects source rows. `aggr='add'` sums messages at their destinations. Our own-state term handles water's empty neighborhood.

We will compare PyG with the explicit `index_add` implementation from earlier lessons using **the same parameters**. This is an equality test between implementations, not a model benchmark.

In [ ]:
class BondSumLayer(MessagePassing):
    def __init__(self, node_dim, edge_dim, hidden=8):
        super().__init__(aggr='add', flow='source_to_target', node_dim=0)
        self.self_map = nn.Linear(node_dim, hidden)
        self.node_map = nn.Linear(node_dim, hidden, bias=False)
        self.edge_map = nn.Linear(edge_dim, hidden, bias=False)

    def forward(self, x, edge_index, edge_attr):
        incoming = self.propagate(edge_index, x=x, edge_attr=edge_attr,
                                  size=(len(x), len(x)))
        return self.self_map(x) + incoming

    def message(self, x_j, edge_attr):
        return torch.relu(self.node_map(x_j) + self.edge_map(edge_attr))

torch.manual_seed(SEED)
layer = BondSumLayer(len(NODE_NAMES), len(EDGE_NAMES))
pyg_h = layer(batch.x, batch.edge_index, batch.edge_attr)
manual_messages = torch.relu(layer.node_map(batch.x[src]) + layer.edge_map(batch.edge_attr))
manual_h = layer.self_map(batch.x) + torch.zeros_like(pyg_h).index_add(0, dst, manual_messages)
torch.testing.assert_close(pyg_h, manual_h, atol=1e-6, rtol=1e-6)

test_x = batch.x.clone().requires_grad_(True)
pyg_gradient = torch.autograd.grad(layer(test_x, batch.edge_index, batch.edge_attr).square().sum(), test_x)[0]
manual_x = batch.x.clone().requires_grad_(True)
manual_msg = torch.relu(layer.node_map(manual_x[src])+layer.edge_map(batch.edge_attr))
manual_out = layer.self_map(manual_x)+manual_x.new_zeros((len(manual_x), 8)).index_add(0, dst, manual_msg)
manual_gradient = torch.autograd.grad(manual_out.square().sum(), manual_x)[0]
torch.testing.assert_close(pyg_gradient, manual_gradient, atol=1e-6, rtol=1e-6)
print('Forward values and input gradients agree with explicit indexed addition.')

## 12.6.5 Use built-in layers deliberately

`GCNConv` performs normalized mixing on a connectivity graph. Its optional scalar `edge_weight` is not a general chemical bond-feature vector. By default it adds self-loops and normalizes the adjacency. For the next check, those choices are explicit:

$$H'=\tilde D^{-1/2}(A+I)\tilde D^{-1/2}XW.$$

$A$ is the adjacency, $I$ adds self-connections, and $\tilde D$ contains row sums of $A+I$. We remove the bias for a compact comparison. [GCNConv API](https://pytorch-geometric.readthedocs.io/en/2.8.0/generated/torch_geometric.nn.conv.GCNConv.html).

In [ ]:
ethanol_data = graphs[0]
gcn = GCNConv(len(NODE_NAMES), 4, bias=False, add_self_loops=True, normalize=True)
gcn_output = gcn(ethanol_data.x, ethanol_data.edge_index)
dense = torch.eye(ethanol_data.num_nodes)
e_src, e_dst = ethanol_data.edge_index
dense[e_dst, e_src] = 1
inverse_sqrt_degree = dense.sum(1).rsqrt()
normalized = inverse_sqrt_degree[:, None] * dense * inverse_sqrt_degree[None, :]
manual_gcn = normalized @ gcn.lin(ethanol_data.x)
torch.testing.assert_close(gcn_output, manual_gcn, atol=1e-6, rtol=1e-6)
print('GCNConv agrees with explicit normalized adjacency.')

### Bond-aware built-in layer: `GINEConv`

GINE extends a graph isomorphism network to consume edge attributes:

$$h_i'=\operatorname{MLP}\left((1+\epsilon)h_i+
\sum_{j\in\mathcal N(i)}\operatorname{ReLU}(h_j+L_e e_{ji})\right).$$

The optional map $L_e$ matches edge width to node width when `edge_dim` is given. The outer MLP is an ordinary PyTorch module. `train_eps=False` keeps $\epsilon$ fixed. This self-term is already part of the operation; do not add self-loop edges casually and double-count it. [GINEConv API](https://pytorch-geometric.readthedocs.io/en/2.8.0/generated/torch_geometric.nn.conv.GINEConv.html) and [pretraining paper introducing this edge-aware variant](https://arxiv.org/abs/1905.12265).

**Research check:** a supposed bond-aware model should actually receive `edge_attr`. The following tensor edit tests sensitivity to that input with fixed weights. It is not a valid chemical bond change because atom/H features are held fixed.

In [ ]:
gine = GINEConv(nn.Sequential(nn.Linear(len(NODE_NAMES), 16), nn.ReLU(), nn.Linear(16, 8)),
                edge_dim=len(EDGE_NAMES), train_eps=False)
gine.eval()
with torch.inference_mode():
    atom_vectors = gine(batch.x, batch.edge_index, batch.edge_attr)
    graph_vectors = global_mean_pool(atom_vectors, batch.batch)
    changed_attr = batch.edge_attr.clone()
    changed_attr[:2, 0] = 0
    changed_attr[:2, 1] = 1  # both directions of ethanol's first edge
    edited_vectors = global_mean_pool(gine(batch.x, batch.edge_index, changed_attr), batch.batch)
assert graph_vectors.shape == (3, 8)
assert not torch.allclose(edited_vectors[0], graph_vectors[0])
torch.testing.assert_close(edited_vectors[1:], graph_vectors[1:])
fig, ax = plt.subplots(figsize=(7, 3.3), layout='constrained')
ax.bar(np.arange(8)-0.18, graph_vectors[0].numpy(), width=0.35, label='Original input')
ax.bar(np.arange(8)+0.18, edited_vectors[0].numpy(), width=0.35, label='Edited bond-feature tensor')
ax.set(xlabel='Readout channel', ylabel='Untrained embedding value',
       title='A real GINEConv uses bond features; other graphs stay unchanged')
ax.legend(fontsize=8)
fig.savefig(OUT / 'gine_bond_input.png', dpi=140)
plt.show()

## 12.6.6 Keep the invariance tests when adopting a library

The library handles indexing, but the full model can still be wrong. Reorder atoms and compare graph readouts. Process molecules separately and in a batch. Neither check establishes chemical accuracy; both are inexpensive ways to catch integration bugs.

In [ ]:
reordered = mol_to_data(Chem.RenumberAtoms(molecules[0], [2, 0, 1]))
with torch.inference_mode():
    old_h = gine(graphs[0].x, graphs[0].edge_index, graphs[0].edge_attr)
    new_h = gine(reordered.x, reordered.edge_index, reordered.edge_attr)
    separate_vectors = torch.stack([gine(g.x, g.edge_index, g.edge_attr).mean(0) for g in graphs])
torch.testing.assert_close(new_h, old_h[[2, 0, 1]], atol=1e-6, rtol=1e-6)
torch.testing.assert_close(separate_vectors, graph_vectors, atol=1e-6, rtol=1e-6)
record = {'versions': VERSIONS, 'feature_schema': FEATURE_SCHEMA,
          'scope': 'Untrained library, routing and gradient checks', 'molecules': smiles,
          'checks': ['Data round-trip', 'target routing', 'edgeless graph', 'manual MessagePassing',
                     'gradient equality', 'manual GCNConv', 'bond sensitivity', 'permutation', 'batch isolation']}
(OUT / 'library_checks.json').write_text(json.dumps(record, indent=2)+'\n', encoding='utf-8')
print('PyG atom permutation and separate/batched inference checks passed.')

## Worked research checklist

Before a large property-model run, use three small records to verify: structures parse under a stated policy; feature widths match the model; targets follow shuffled graphs; isolated atoms survive batching; a bond-aware layer consumes bond features; the graph output is invariant to storage order. Passing these checks saves debugging time but still leaves the data split, target definition and scientific assessment to design.

## Exercises and suggested answers

1. A loader contains 32 molecules and 640 atoms. How many graph targets should it return? What is the first axis of its atom-output matrix?
2. Why can water have `num_nodes=1` and `num_edges=0` without being invalid?
3. What does PyG supply to a custom `message(x_j, edge_attr)` method? Where is the result aggregated?
4. Why should a seven-column bond descriptor not be passed as `GCNConv`'s scalar `edge_weight`?
5. How would you diagnose predictions that change when batch companions change?

<details><summary>Answers</summary>

1. 32 targets; 640 atom rows. Graph pooling changes the first dimension to 32.
2. This heavy-atom representation stores water's oxygen and its attached H count, with no heavy-atom bond.
3. Source-node features and the corresponding directed-edge attributes. With source-to-target flow, aggregation groups messages at destination nodes.
4. They have different semantics and shapes. Use a layer designed to consume edge attributes, or explicitly define a scientifically justified scalar weighting.
5. Check edge offsets, graph membership, normalization over the whole batch, dropout/evaluation mode, and readout grouping. Repeat separate/batched and permutation tests on the complete model.

</details>

[Next: 12.7 — train a PyG model on measured solubility](Chapter12_Part7.ipynb) · [Course guide](docs/course-guide.md) · [Course index](Readme.md)